# B2B Pricing - Cohort-Based CPR Scoring

Entry-point notebook for the **cohort-based CPRScore** methodology.

Uses `B2B_functions_cohort_score.py`, retaining the loan-level cohort benchmarks from `Analytics_Rpt.md.B2B_revamp_cohorts` instead of the pre-stored `ModelCPR` column.

**Key differences from the original report:**
- Explicit scoring and funding windows remain user-controlled.
- CPRScore retains the cohort-based benchmark and unmapped-loan defaults.
- Performance inputs now use `Analytics_Rpt.md.B2BPricingHist2`, with PR, VI, and GU excluded, matching the updated BU report.
- `CohortRecap`, `ActualRecap`, and `RecapScore` retain the BU weighted formulas and score defaults, using the updated source values.
- The validation section compares recapture fields against both monthly reporter modules on identical loaded inputs.

Use the **Python (B2B)** kernel (env: `B2B`).

## 0. USER INPUT: Report Month and Output File Path

In [1]:
from datetime import date

# Report parameters:
pi_Product        = 'All'
pi_BeginScoreDt   = date(2025, 10, 31)
pi_EndScoreDt     = date(2026, 5, 31)
pi_BeginRefFundDt = None
pi_EndRefFundDt   = None

file_path = r"C:\Users\kanshash\OneDrive - Onity Group Inc\Desktop\Dev\B2B_cohortScore_update\B2B_reporting_raw"
report_tag = pi_EndScoreDt.strftime('%Y%m%d') if pi_EndScoreDt is not None else date.today().strftime('%Y%m%d')
file_name = f"B2B_cohort_score_{report_tag}_upd.xlsx"

print(f"Score window  : {pi_BeginScoreDt or '1900-01-01'}  ->  {pi_EndScoreDt or 'all history'}")
print(f"Funding window: {pi_BeginRefFundDt or '(default)'}  ->  {pi_EndRefFundDt or '(default)'}")
print(f"Output file   : {file_path}\\{file_name}")

Score window  : 2025-10-31  ->  2026-05-31
Funding window: (default)  ->  (default)
Output file   : C:\Users\kanshash\OneDrive - Onity Group Inc\Desktop\Dev\B2B_cohortScore_update\B2B_reporting_raw\B2B_cohort_score_20260531_upd.xlsx


## 1. Imports and module reload

In [ ]:
import os, sys

import numpy as np
import pandas as pd

# Make the local 'functions' folder importable
FUNCTIONS_DIR = os.path.join(os.getcwd(), 'functions')
if FUNCTIONS_DIR not in sys.path:
    sys.path.insert(0, FUNCTIONS_DIR)

%load_ext autoreload
%autoreload 2

import B2B_functions_cohort_score as bf
from B2B_functions_cohort_score import (
    sql_connection,
    B2BPricingReporter,
    BandClassifier,
    DateHelpers,
    DEFAULT_METRIC_ORDER,
    PRODUCT_LOOP,
)

print('B2B_functions_cohort_score_logfallback loaded from:', bf.__file__)

B2B_functions_cohort_score_logfallback loaded from: c:\Users\kanshash\OneDrive - Onity Group Inc\Desktop\Dev\B2B_cohortScore_update\repo\python\functions\B2B_functions_cohort_score_logfallback.py


In [ ]:
import importlib
bf = importlib.reload(bf)

from B2B_functions_cohort_score import (
    sql_connection,
    B2BPricingReporter,
    BandClassifier,
    DateHelpers,
    DEFAULT_METRIC_ORDER,
    PRODUCT_LOOP,
)

## 2. Connect to SQL Server

In [4]:
engine = sql_connection(
    server='VMC2VMDBRADC01P',
    database='Analytics_Rpt',
    driver='ODBC Driver 17 for SQL Server',
)
engine

Engine(mssql+pyodbc:///?odbc_connect=DRIVER%3D%7BODBC+Driver+17+for+SQL+Server%7D%3BSERVER%3DVMC2VMDBRADC01P%3BDATABASE%3DAnalytics_Rpt%3BTrusted_Connection%3Dyes)

## 3. Load source frames

`reporter.load(begin_score_dt=..., end_score_dt=..., begin_fund_dt=..., end_fund_dt=...)` runs four queries and caches the DataFrames.

Key change vs. the original: pass explicit funding and scoring windows instead of a single auto-derived report month:
- `begin_fund_dt` / `end_fund_dt`   = `pi_BeginRefFundDt` / `pi_EndRefFundDt`
- `begin_score_dt` / `end_score_dt` = `pi_BeginScoreDt` / `pi_EndScoreDt`
- Performance data is joined to `B2B_revamp_cohorts` to assign `Cohort_ID` per loan-month.

In [6]:
reporter = B2BPricingReporter(engine).load(
    begin_score_dt=pi_BeginScoreDt,
    end_score_dt=pi_EndScoreDt,
    begin_fund_dt=pi_BeginRefFundDt,
    end_fund_dt=pi_EndRefFundDt,
)

print('BeginScoreDt    :', reporter.begin_score_dt)
print('EndScoreDt      :', reporter.end_score_dt)
print('BeginFundDt     :', reporter.begin_fund_dt)
print('EndFundDt       :', reporter.end_fund_dt)
print('volume_all      :', reporter.volume_all.shape)
print('perform_all     :', reporter.perform_all.shape)
print('client_seller   :', reporter.client_seller.shape)
print('output_template :', reporter.output_template.shape)

# Cohort coverage for the report month only (what matters for CPRScore)
# perform_all spans all historical months; coverage % is only meaningful for the report month.
import datetime
_target = pi_EndScoreDt or reporter.perform_all['AsofDt'].apply(
    lambda x: x.date() if hasattr(x, 'date') else x).max()
_pa_rm = reporter.perform_all[
    reporter.perform_all['AsofDt'].apply(
        lambda x: (x.date() if hasattr(x, 'date') else x) == _target)
]
_rm_total  = _pa_rm['UniqueLoanCt'].sum()
_rm_mapped = _pa_rm[_pa_rm['Cohort_ID'].notna()]['UniqueLoanCt'].sum()
print(f'\nCohort coverage for {_target}: {_rm_mapped:,} / {_rm_total:,} loan-months '
      f'({100*_rm_mapped/_rm_total:.1f}%) have a Cohort_ID')
print(f'(perform_all spans {reporter.perform_all["AsofDt"].nunique()} months total — '
      f'cohorts only exist for months >= 2023-01-31)')


Report as of  : current
Funding window: 2025-11-01  →  2026-10-31
Score window  : 2025-10-31  →  2026-05-31
BeginScoreDt    : 2025-10-31
EndScoreDt      : 2026-05-31
BeginFundDt     : 2025-11-01
EndFundDt       : 2026-10-31
volume_all      : (148490, 18)
perform_all     : (2560677, 29)
client_seller   : (806, 12)
output_template : (118, 4)

Cohort coverage for 2026-05-31: 378,543 / 378,730 loan-months (100.0%) have a Cohort_ID
(perform_all spans 8 months total — cohorts only exist for months >= 2023-01-31)


## 4. Data Quality — quick checks

In [7]:

# Cohort coverage for the REPORT MONTH only — this is what drives CPRScore
import datetime
_target = pi_EndScoreDt or reporter.perform_all['AsofDt'].apply(
    lambda x: x.date() if hasattr(x, 'date') else x).max()
_pa_rm = reporter.perform_all[
    reporter.perform_all['AsofDt'].apply(
        lambda x: (x.date() if hasattr(x, 'date') else x) == _target)
].copy()
_pa_rm['Cohort_Mapped'] = _pa_rm['Cohort_ID'].notna()
coh_dist = (
    _pa_rm.groupby('Cohort_Mapped', dropna=False)
    .agg(LoanMonths=('LoanCt', 'sum'), UPB=('BeginPrinBal', 'sum'))
    .reset_index()
)
coh_dist['Cohort_Mapped'] = coh_dist['Cohort_Mapped'].map(
    {True: 'Cohort-assigned (GSE/GNMA)', False: 'PLS / unmapped (no CPRScore)'})
coh_dist['UPB'] = coh_dist['UPB'].apply(lambda x: f'${x:,.0f}')
print(f'Cohort coverage for {_target} (report month only):')
coh_dist


Cohort coverage for 2026-05-31 (report month only):


,Cohort_Mapped,LoanMonths,UPB
0,PLS / unmapped (no CPRScore),187,"$53,887,586"
1,Cohort-assigned (GSE/GNMA),378543,"$120,172,529,292"


In [8]:
# Top clients by volume (sanity)
(
    reporter.volume_all
    .groupby('Client', dropna=False)['FundedLoanAmt']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .apply(lambda x: f'${x:,.0f}')
    .rename('FundedLoanAmt')
)

Client
ZILLOW HOME LOANS LLC                    $3,780,932,472
LENNAR MORTGAGE LLC                      $2,578,481,014
MORTGAGE RESEARCH CENTER LLC             $1,856,171,856
AMERICAN PACIFIC MORTGAGE CORPORATION    $1,708,612,299
AVERRA FINANCIAL INC (SS 196043)         $1,702,862,658
PLAZA HOME MORTGAGE INC                  $1,690,749,165
PRIMELENDING A PLAINSCAPITAL COMPANY     $1,187,012,704
GUARANTEED RATE INC                      $1,101,327,978
KIND LENDING LLC                           $907,633,774
LOANDEPOTCOM LLC                           $797,198,978
Name: FundedLoanAmt, dtype: object

## 5. Build `#AllMetrics`

Produces the primary AllMetrics table for each product cut.
Valid `METRIC_ORDER` values: `'Total'`, `'State'`, `'FICO'`, `'LTV'`, `'DTI'`, `'Interest Rate'`, `'Product'`, `'Avg Loan Bal'`, `'Channel'`, `'Client'`.

In [9]:
METRIC_ORDER = ['Total', 'State', 'FICO', 'LTV', 'DTI', 'Interest Rate',
                'Product', 'Avg Loan Bal', 'Channel', 'Client']

def normalize_report_cpr_columns(frame):
    return frame.rename(columns={
        'ActualCPR': 'Client_ActualCPR',
        'ModelCPR': 'Cohort_ActualCPR',
    })

all_metrics = normalize_report_cpr_columns(
    reporter.run_all_metrics(METRIC_ORDER, product=pi_Product)
)
all_metrics.head(20)

,MetricOrder,SortOrder,ProductCategory,MetricName,MetricValue,BeginFundDt,EndFundDt,LoanCt,FundedLoanAmt,VolumeRank,...,Account Manager_West,Account Manager_Central,Account Manager_NULL,ClientID_Delegated - Best Efforts,ClientID_Delegated - Mandatory,ClientID_Direct,ClientID_FHLMC CRX,ClientID_FNMA SMP,ClientID_GNMA PIIT,ClientID_Non-delegated
0,1,1,All,Total,Total,2025-11-01,2026-10-31,155071.0,5.285813e+10,1.0,...,None,None,None,None,None,None,None,None,None,None
1,2,1,All,State,AK,2025-11-01,2026-10-31,230.0,8.378479e+07,49.0,...,None,None,None,None,None,None,None,None,None,None
2,2,2,All,State,AL,2025-11-01,2026-10-31,2535.0,6.905712e+08,28.0,...,None,None,None,None,None,None,None,None,None,None
3,2,3,All,State,AR,2025-11-01,2026-10-31,2140.0,5.960643e+08,29.0,...,None,None,None,None,None,None,None,None,None,None
4,2,4,All,State,AZ,2025-11-01,2026-10-31,4115.0,1.541888e+09,11.0,...,None,None,None,None,None,None,None,None,None,None
5,2,5,All,State,CA,2025-11-01,2026-10-31,6815.0,3.582568e+09,3.0,...,None,None,None,None,None,None,None,None,None,None
6,2,6,All,State,CO,2025-11-01,2026-10-31,4093.0,1.862049e+09,6.0,...,None,None,None,None,None,None,None,None,None,None
7,2,7,All,State,CT,2025-11-01,2026-10-31,695.0,2.656439e+08,35.0,...,None,None,None,None,None,None,None,None,None,None
8,2,8,All,State,DC,2025-11-01,2026-10-31,206.0,1.082195e+08,45.0,...,None,None,None,None,None,None,None,None,None,None
9,2,9,All,State,DE,2025-11-01,2026-10-31,686.0,2.310841e+08,36.0,...,None,None,None,None,None,None,None,None,None,None


In [10]:
# Quick check: Total row CPRScore should be 1.0
cols = ['MetricValue', 'Client_ActualCPR', 'Cohort_ActualCPR', 'CPRScore']
total_row = all_metrics[all_metrics['MetricValue'] == 'Total'][cols]
print('Total row scores:')
total_row

Total row scores:


,MetricValue,Client_ActualCPR,Cohort_ActualCPR,CPRScore
0,Total,0.126568,0.126568,1.0


### Build AllMetrics for every product in the loop

In [11]:
all_metrics_by_product = {
    prod: normalize_report_cpr_columns(
        reporter.run_all_metrics(METRIC_ORDER, product=prod)
    )
    for prod in PRODUCT_LOOP
}
{k: v.shape for k, v in all_metrics_by_product.items()}

{'All': (924, 35),
 'Conventional': (898, 35),
 'FHA': (650, 35),
 'VA': (613, 35),
 'USDA': (165, 35),
 'Govt': (695, 35)}

## 6. Client × State Summary

In [12]:
METRIC_ORDER_CS = ['Client', 'PropState']
client_state = normalize_report_cpr_columns(reporter.build_groupby_summary(METRIC_ORDER_CS))
print(f'Client × State: {client_state.shape}')
client_state.head(20)

Client × State: (7426, 32)


,Client,PropState,BeginFundDt,EndFundDt,LoanCt,FundedLoanAmt,VolumeRank,GovtPct,WAFICO,WALTV,...,Account Manager_West,Account Manager_Central,Account Manager_NULL,ClientID_Delegated - Best Efforts,ClientID_Delegated - Mandatory,ClientID_Direct,ClientID_FHLMC CRX,ClientID_FNMA SMP,ClientID_GNMA PIIT,ClientID_Non-delegated
0,1ST 2ND MORTGAGE COMPANY OF NJ INC,FL,2025-11-01,2026-10-31,1,750000.00,4691,0.000000,784.000000,0.458720,...,None,None,None,None,74123,None,None,None,None,None
1,1ST 2ND MORTGAGE COMPANY OF NJ INC,NJ,2025-11-01,2026-10-31,6,4161169.72,2094,0.313010,710.284191,0.793377,...,None,None,None,None,74123,None,None,None,None,None
2,1ST 2ND MORTGAGE COMPANY OF NJ INC,NY,2025-11-01,2026-10-31,7,4076649.42,2128,0.645240,696.049226,0.811433,...,None,None,None,None,74123,None,None,None,None,None
3,1ST FINANCIAL INC,FL,2025-11-01,2026-10-31,22,9742725.01,1181,0.317392,738.219333,0.825190,...,None,None,None,74533,74533,None,None,None,None,None
4,1ST NATIONAL BANK,IN,2025-11-01,2026-10-31,1,593750.00,5160,0.000000,755.000000,0.950000,...,None,Tracey Dace,None,74930,None,None,108581,None,None,None
5,1ST NATIONAL BANK,OH,2025-11-01,2026-10-31,8,2020538.75,3053,1.000000,706.520402,0.948205,...,None,Tracey Dace,None,74930,None,None,108581,None,None,None
6,1ST PRIORITY MORTGAGE INC,CT,2025-11-01,2026-10-31,1,314022.70,6450,0.000000,641.000000,0.700000,...,None,None,None,74225,74225,None,214742,None,None,None
7,1ST PRIORITY MORTGAGE INC,NJ,2025-11-01,2026-10-31,4,3339550.56,2392,0.000000,787.247518,0.682423,...,None,None,None,74225,74225,None,214742,None,None,None
8,1ST PRIORITY MORTGAGE INC,NY,2025-11-01,2026-10-31,36,12291610.93,968,0.193363,726.799913,0.857882,...,None,None,None,74225,74225,None,214742,None,None,None
9,1ST PRIORITY MORTGAGE INC,PA,2025-11-01,2026-10-31,1,286150.00,6611,0.000000,700.000000,0.970000,...,None,None,None,74225,74225,None,214742,None,None,None


## 7. Client × State × Product Summary

In [13]:
METRIC_ORDER_CSP = ['Client', 'PropState', 'ProductCategory']
client_state_product = normalize_report_cpr_columns(
    reporter.build_groupby_summary(METRIC_ORDER_CSP)
)
print(f'Client × State × Product: {client_state_product.shape}')
client_state_product.head(20)

Client × State × Product: (12285, 33)


,Client,PropState,ProductCategory,BeginFundDt,EndFundDt,LoanCt,FundedLoanAmt,VolumeRank,GovtPct,WAFICO,...,Account Manager_West,Account Manager_Central,Account Manager_NULL,ClientID_Delegated - Best Efforts,ClientID_Delegated - Mandatory,ClientID_Direct,ClientID_FHLMC CRX,ClientID_FNMA SMP,ClientID_GNMA PIIT,ClientID_Non-delegated
0,1ST 2ND MORTGAGE COMPANY OF NJ INC,FL,Conventional,2025-11-01,2026-10-31,1,750000.00,6804,0.0,784.000000,...,None,None,None,None,74123,None,None,None,None,None
1,1ST 2ND MORTGAGE COMPANY OF NJ INC,NJ,Conventional,2025-11-01,2026-10-31,4,2858681.72,2978,0.0,739.093723,...,None,None,None,None,74123,None,None,None,None,None
2,1ST 2ND MORTGAGE COMPANY OF NJ INC,NJ,FHA,2025-11-01,2026-10-31,1,716777.00,6985,1.0,652.000000,...,None,None,None,None,74123,None,None,None,None,None
3,1ST 2ND MORTGAGE COMPANY OF NJ INC,NJ,VA,2025-11-01,2026-10-31,1,585711.00,7779,1.0,641.000000,...,None,None,None,None,74123,None,None,None,None,None
4,1ST 2ND MORTGAGE COMPANY OF NJ INC,NY,Conventional,2025-11-01,2026-10-31,2,1446231.29,4647,0.0,739.632147,...,None,None,None,None,74123,None,None,None,None,None
5,1ST 2ND MORTGAGE COMPANY OF NJ INC,NY,FHA,2025-11-01,2026-10-31,5,2630418.13,3165,1.0,672.086882,...,None,None,None,None,74123,None,None,None,None,None
6,1ST FINANCIAL INC,FL,Conventional,2025-11-01,2026-10-31,14,6650457.49,1590,0.0,752.062960,...,None,None,None,74533,74533,None,None,None,None,None
7,1ST FINANCIAL INC,FL,FHA,2025-11-01,2026-10-31,8,3092267.52,2840,1.0,708.446213,...,None,None,None,74533,74533,None,None,None,None,None
8,1ST NATIONAL BANK,IN,Conventional,2025-11-01,2026-10-31,1,593750.00,7737,0.0,755.000000,...,None,Tracey Dace,None,74930,None,None,108581,None,None,None
9,1ST NATIONAL BANK,OH,FHA,2025-11-01,2026-10-31,6,1534836.84,4465,1.0,685.802890,...,None,Tracey Dace,None,74930,None,None,108581,None,None,None


## 8. CPRScore Audit Table (Exec-facing)

Built by `functions/B2B_functions_cpr_audit.py` — a standalone audit of the cohort calculations.

### Detail columns in `cprAuditLong`

Each row represents one `Client × Cohort_ID × Rule_bucket` cell.

| Column | Meaning |
|---|---|
| `ReportAsofDt` | Month-end date as of which the analysis was performed |
| `AsofDt` | Current calendar month's month-end date when the notebook runs |
| `Rule_bucket` | Cohort rule assigned to the cohort |
| `Client_LoanCt` | Client's loan-month contribution to the cohort/rule bucket |
| `Client_BeginPrinBal` / `Client_EndPrinBal` | Client's balance contribution |
| `Cohort_LoanCt` | Total loan-month count across the cohort |
| `Cohort_BeginPrinBal` / `Cohort_EndPrinBal` | Total balances across the cohort |
| `Client_ActualCPR` | Client-wide balance-rollup ActualCPR |
| `Client_x_Cohort_ActualCPR` | Client's ActualCPR within this cohort |
| `Cohort_ActualCPR` | Client-level cohort-balance rollup used as the CPRScore denominator |
| `Cohort_x_Cohort_ActualCPR` | Cohort-level benchmark CPR |
| `CPRScore` | `Client_ActualCPR / Cohort_ActualCPR` |

In [14]:
import B2B_functions_cpr_audit as bfa
importlib.reload(bfa)

# Audit module defaults to the base module's cutoff; pass the report's cutoff so both agree.
audit = bfa.build_cpr_audit(reporter, product='All', decimals=5,
                            min_loans=bf.MIN_COHORT_LOANS)

cpr_audit_wide = audit['wide']
cpr_audit_long = audit['long']

print(f'\nwide : {cpr_audit_wide.shape[0]:,} rows x {cpr_audit_wide.shape[1]:,} cols')
print(f'long : {cpr_audit_long.shape[0]:,} rows x {cpr_audit_long.shape[1]:,} cols')


c:\Users\kanshash\OneDrive - Onity Group Inc\Desktop\Dev\B2B_cohortScore_update\repo\python\functions\B2B_functions_cpr_audit.py:313: RuntimeWarning: invalid value encountered in divide
  client_score = np.where(client_model > 0, client_actual / client_model, 1.0)


CPR audit [All]  clients=1,059  cohorts=106  wide shape=(1060, 557)
  tie-out (exact vs published): 1,059 OK / 0 MISMATCH  (max |diff| = 1.51e-13, tol = 1e-09)
  max |rounding drift| at 5dp: 0.00e+00 CPRScore points

wide : 1,060 rows x 557 cols
long : 33,520 rows x 19 cols


In [15]:
# Validation 1 — audit CPRScore vs. rawTotal Client rows
rpt_client = (
    all_metrics.loc[all_metrics['MetricName'] == 'Client',
                    ['MetricValue', 'Client_ActualCPR', 'Cohort_ActualCPR', 'CPRScore']]
    .rename(columns={'MetricValue': 'Client',
                     'Client_ActualCPR': 'Client_ActualCPR_rawTotal',
                     'Cohort_ActualCPR': 'Cohort_ActualCPR_rawTotal',
                     'CPRScore': 'CPRScore_rawTotal'})
)
chk = cpr_audit_wide.merge(rpt_client, on='Client', how='inner')
chk['Diff_vs_rawTotal'] = chk['CPRScore_Exact'].round(6) - chk['CPRScore_rawTotal']
print(f'Clients matched to rawTotal      : {len(chk):,}')
print(f'Max |diff| vs rawTotal           : {chk["Diff_vs_rawTotal"].abs().max():.2e}')
print(f'Clients off rawTotal by > 1e-6   : {(chk["Diff_vs_rawTotal"].abs() > 1e-6).sum():,}')

Clients matched to rawTotal      : 772
Max |diff| vs rawTotal           : 0.00e+00
Clients off rawTotal by > 1e-6   : 0


In [16]:
# Validation 2 — ModelCPR uses summed cohort balances directly
_body = cpr_audit_long[cpr_audit_long['Cohort_ID'].notna()].copy()
long_scored = cpr_audit_long[cpr_audit_long['Included_Flag'] == 1].copy()
expected = (long_scored.drop_duplicates(['Client', 'Cohort_ID'])
            .groupby('Client', as_index=False)
            .agg(Begin=('Cohort_BeginPrinBal', 'sum'), End=('Cohort_EndPrinBal', 'sum')))
expected['Cohort_ActualCPR_Expected'] = 1 - (expected['End'] / expected['Begin']) ** 12
check = (_body[['Client', 'Cohort_ActualCPR']].drop_duplicates('Client')
         .merge(expected[['Client', 'Cohort_ActualCPR_Expected']], on='Client', how='inner'))
check['Diff'] = check['Cohort_ActualCPR'] - check['Cohort_ActualCPR_Expected']
print(f'Clients checked             : {len(check):,}')
print(f'Max |Cohort_ActualCPR diff| : {check["Diff"].abs().max():.2e}')
print(f'Clients outside 1e-9        : {(check["Diff"].abs() > 1e-9).sum():,}')

Clients checked             : 975
Max |Cohort_ActualCPR diff| : 0.00e+00
Clients outside 1e-9        : 0


## 8A. Client FUN Classification

Built by `functions/B2B_functions_fun_classification.py`, using the same rule as `B2B_Client_Score_Stability_Analysis.ipynb`. **Z** is `TotalScore` on the `All / Total / Total` row of `rawTotal`:
- **F**: `TotalScore >= Z x 1.25`
- **U**: `TotalScore < Z x 0.75` (strict)
- **N**: otherwise

`FUN_Classification` is added as the last column of `rawTotal` and populated only for `MetricName = 'Client'`. Existing columns are not moved. `all_metrics` itself is unchanged; the export uses the tagged copy `all_metrics_fun`.

In [17]:
import B2B_functions_fun_classification as bfun
importlib.reload(bfun)

all_metrics_fun, fun_thresholds = bfun.add_fun_classification(all_metrics)
client_scores_final = bfun.client_score_table(all_metrics_fun)

assert all_metrics_fun.drop(columns=bfun.FUN_COLUMN).equals(all_metrics), 'Existing rawTotal values changed'
assert all_metrics_fun.loc[all_metrics_fun['MetricName'].ne('Client'), bfun.FUN_COLUMN].isna().all()
assert client_scores_final[bfun.FUN_COLUMN].isin(['F', 'U', 'N']).all()
assert len(client_scores_final) == all_metrics['MetricName'].eq('Client').sum()

print(f"Z = {fun_thresholds['Z']:.6f} | F >= {fun_thresholds['Upper threshold']:.6f} | "
      f"U < {fun_thresholds['Lower threshold']:.6f}")
fun_summary = (client_scores_final.groupby(bfun.FUN_COLUMN)
               .agg(Clients=('MetricValue', 'size'), FundedLoanAmt=('FundedLoanAmt', 'sum'))
               .reindex(['F', 'N', 'U'], fill_value=0))
fun_summary['Client %'] = fun_summary['Clients'] / fun_summary['Clients'].sum()
fun_summary['Funded %'] = fun_summary['FundedLoanAmt'] / fun_summary['FundedLoanAmt'].sum()
fun_summary.style.format({'Clients': '{:,}', 'FundedLoanAmt': '${:,.0f}',
                          'Client %': '{:.1%}', 'Funded %': '{:.1%}'})

Z = 1.002066 | F >= 1.252582 | U < 0.751549


,Clients,FundedLoanAmt,Client %,Funded %
FUN_Classification,,,,
F,314,"$23,698,893,800",39.0%,44.8%
N,262,"$13,410,947,899",32.5%,25.4%
U,230,"$15,748,289,864",28.5%,29.8%


## 9. Export to Excel

In [18]:
import os

full_path = os.path.join(file_path, file_name)
os.makedirs(file_path, exist_ok=True)

with pd.ExcelWriter(full_path, engine='xlsxwriter') as writer:

    # 1) rawTotal (with FUN_Classification on Client rows)
    all_metrics_fun.fillna('NULL').to_excel(writer, sheet_name='rawTotal', index=False)

    # 2) rawConv
    all_metrics_by_product['Conventional'].fillna('NULL') \
        .to_excel(writer, sheet_name='rawConv', index=False)

    # 3) rawFHA
    all_metrics_by_product['FHA'].fillna('NULL') \
        .to_excel(writer, sheet_name='rawFHA', index=False)

    # 4) rawVA
    all_metrics_by_product['VA'].fillna('NULL') \
        .to_excel(writer, sheet_name='rawVA', index=False)

    # 5) rawUSDA
    all_metrics_by_product['USDA'].fillna('NULL') \
        .to_excel(writer, sheet_name='rawUSDA', index=False)

    # 6) rawGovt
    all_metrics_by_product['Govt'].fillna('NULL') \
        .to_excel(writer, sheet_name='rawGovt', index=False)

    # 7) rawClientState
    client_state.fillna('NULL').to_excel(writer, sheet_name='rawClientState', index=False)

    # 8) rawClientStateProduct
    client_state_product.fillna('NULL').to_excel(writer, sheet_name='rawClientStateProduct', index=False)

    # 9) cprAuditWide — Exec-facing CPRScore audit (one row per Client)
    cpr_audit_wide.to_excel(writer, sheet_name='cprAuditWide', index=False)

    # 10) cprAuditLong — cell-level detail incl. cells excluded from the score
    cpr_audit_long.fillna('NULL').to_excel(writer, sheet_name='cprAuditLong', index=False)

print(f'Excel saved: {full_path}')


Excel saved: C:\Users\kanshash\OneDrive - Onity Group Inc\Desktop\Dev\B2B_cohortScore_update\B2B_reporting_raw\B2B_cohort_score_20260531_upd.xlsx


In [19]:
# Read-back check: rawTotal keeps its original columns in place and carries FUN_Classification last
# keep_default_na=False so the literal 'NULL' text is not parsed back to NaN
raw_total_check = pd.read_excel(full_path, sheet_name='rawTotal', keep_default_na=False)
assert raw_total_check.columns[:-1].tolist() == all_metrics.columns.tolist()
assert raw_total_check.columns[-1] == bfun.FUN_COLUMN
exported_fun = raw_total_check.loc[raw_total_check['MetricName'].eq('Client'), bfun.FUN_COLUMN]
assert exported_fun.tolist() == client_scores_final[bfun.FUN_COLUMN].tolist()
assert raw_total_check.loc[raw_total_check['MetricName'].ne('Client'), bfun.FUN_COLUMN].eq('NULL').all()
print(f'rawTotal read-back OK: {len(exported_fun):,} Client rows tagged '
      f"{exported_fun.value_counts().reindex(['F', 'N', 'U'], fill_value=0).to_dict()}")

rawTotal read-back OK: 806 Client rows tagged {'F': 314, 'N': 262, 'U': 230}


## 10. Push CPRScore Audit to SQL (disabled by default)

`PUSH_CPR_AUDIT_TO_SQL = False` prepares the audit frame for diagnostics without writing to SQL. Set it to `True` only after the Excel has been verified and replacement of the shared table has been explicitly approved.

`if_exists='replace'` drops and rebuilds the table. Use `'append'` instead only when loading additional report months into an established table is intended.

In [20]:
PUSH_CPR_AUDIT_TO_SQL = False
AUDIT_TABLE = 'B2B_CPRScore_Audit'
SCHEMA = 'md'

required_audit_columns = {'ReportAsofDt', 'AsofDt', 'Cohort_ID'}
missing_audit_columns = required_audit_columns.difference(cpr_audit_long.columns)
assert not missing_audit_columns, f'Missing audit columns: {sorted(missing_audit_columns)}'

cpr_audit_long_sql = cpr_audit_long[
    cpr_audit_long['Rule_bucket'].notna()
].copy()

if PUSH_CPR_AUDIT_TO_SQL:
    cpr_audit_long_sql.to_sql(
        name=AUDIT_TABLE,
        con=engine,
        schema=SCHEMA,
        if_exists='replace',
        index=False,
        chunksize=100,
        method=None,
    )
    print(f'Written: Analytics_Rpt.{SCHEMA}.{AUDIT_TABLE} '
          f'({cpr_audit_long_sql.shape})')
else:
    print(f'SQL replacement skipped: Analytics_Rpt.{SCHEMA}.{AUDIT_TABLE}; '
          f'{len(cpr_audit_long_sql):,} rows prepared for diagnostics only.')

SQL replacement skipped: Analytics_Rpt.md.B2B_CPRScore_Audit; 33,418 rows prepared for diagnostics only.


In [21]:
# Diagnostic only — final audit schema and date validation
print('Audit columns:', cpr_audit_long.columns.tolist())
print('ReportAsofDt values:', cpr_audit_long['ReportAsofDt'].drop_duplicates().tolist())
print('AsofDt values:', cpr_audit_long['AsofDt'].drop_duplicates().tolist())
print('Duplicate column labels:', cpr_audit_long.columns[cpr_audit_long.columns.duplicated()].tolist())
print('SQL push rows:', len(cpr_audit_long_sql), 'null Cohort_ID rows:', int(cpr_audit_long_sql['Cohort_ID'].isna().sum()))

Audit columns: ['ReportAsofDt', 'AsofDt', 'ProductCategory', 'Client', 'Cohort_ID', 'Rule_bucket', 'Client_LoanCt', 'Client_BeginPrinBal', 'Client_EndPrinBal', 'Cohort_LoanCt', 'Cohort_BeginPrinBal', 'Cohort_EndPrinBal', 'Client_ActualCPR', 'Client_x_Cohort_ActualCPR', 'Cohort_ActualCPR', 'Cohort_x_Cohort_ActualCPR', 'CPRScore', 'Included_Flag', 'ExclusionReason']
ReportAsofDt values: [datetime.date(2026, 5, 31)]
AsofDt values: [datetime.date(2026, 11, 30)]
Duplicate column labels: []
SQL push rows: 33418 null Cohort_ID rows: 0


## 11. Push Client Scores to SQL (disabled by default)

Writes `client_scores_final` to `Analytics_Rpt.md.B2B_ClientScore_final`. This contains the `rawTotal` rows with `MetricName = 'Client'`, Excel columns D:X (`MetricName` .. `TotalScore`), and `FUN_Classification`.

`PUSH_CLIENT_SCORES_TO_SQL = False` prepares the frame without writing to SQL. Set it to `True` only after the Excel has been verified. `if_exists='replace'` drops and rebuilds the table; use `'append'` only when loading additional report runs is intended.

In [22]:
PUSH_CLIENT_SCORES_TO_SQL = False
CLIENT_SCORE_TABLE = 'B2B_ClientScore_final'
CLIENT_SCORE_SCHEMA = 'md'

assert client_scores_final.columns[0] == 'MetricName'
assert client_scores_final.columns[-2:].tolist() == ['TotalScore', bfun.FUN_COLUMN]
assert client_scores_final['MetricName'].eq('Client').all()
assert not client_scores_final['MetricValue'].duplicated().any(), 'Duplicate clients'

if PUSH_CLIENT_SCORES_TO_SQL:
    client_scores_final.to_sql(
        name=CLIENT_SCORE_TABLE,
        con=engine,
        schema=CLIENT_SCORE_SCHEMA,
        if_exists='replace',
        index=False,
        chunksize=100,
        method=None,
    )
    print(f'Written: Analytics_Rpt.{CLIENT_SCORE_SCHEMA}.{CLIENT_SCORE_TABLE} '
          f'({client_scores_final.shape})')
else:
    print(f'SQL replacement skipped: Analytics_Rpt.{CLIENT_SCORE_SCHEMA}.{CLIENT_SCORE_TABLE}; '
          f'{len(client_scores_final):,} rows x {client_scores_final.shape[1]} cols prepared.')
    print('Columns:', client_scores_final.columns.tolist())

SQL replacement skipped: Analytics_Rpt.md.B2B_ClientScore_final; 806 rows x 22 cols prepared.
Columns: ['MetricName', 'MetricValue', 'BeginFundDt', 'EndFundDt', 'LoanCt', 'FundedLoanAmt', 'VolumeRank', 'GovtPct', 'WAFICO', 'WALTV', 'WADTI', 'WAC', 'CPRLoanct', 'Cohort_ActualCPR', 'Client_ActualCPR', 'CPRScore', 'RecapLoanct', 'CohortRecap', 'ActualRecap', 'RecapScore', 'TotalScore', 'FUN_Classification']
